# 01 – Gagnavinnsla

Í þessu skjali sækjum við gögn úr Íslensku kosningarannsókninni (ÍSKOS) 2024, skilgreinum útkomubreytuna (flokksval í Alþingiskosningunum 2024) og veljum forspárbreytur í fjórum breytuhópum: **félagslega stöðu**, **málefnaviðhorf**, **frammistöðu stjórnvalda** og **málefnaeign**.

Að því loknu skiptum við gögnunum í **þjálfunarhluta (80%)** og **prófunarhluta (20%)** og vistum niðurstöðurnar á disk. Annað skjalið (`02_models.ipynb`) þjálfar líkönin á þessum sömu gögnum og þriðja skjalið (`03_shap.ipynb`) túlkar besta líkanið með SHAP sem er aðferð til að útskýra niðurstöður líkana.

**Markmið þessa skjals** er að búa til eftirfarandi skrár í möppunum `data/processed/` og `figures/`:

- `train.parquet`, `test.parquet` — hreinsaður þjálfunar- og prófunarhluti
- `feature_groups.json` — hvaða breytur tilheyra hvaða breytuhópi
- `target_labels.json` — kóði → íslenskt flokksheiti og breytukóði → íslensk breytulýsing (notað í myndum seinna)

In [ ]:
# Standard library
from __future__ import annotations

import json
from pathlib import Path

# Third-party
import numpy as np
import pandas as pd
import pyreadstat
from sklearn.model_selection import train_test_split

# Notebook-level config: keep reproducibility and a single source of truth for paths
RANDOM_STATE = 42  # used for the stratified train/test split
TEST_SIZE = 0.20

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = PROJECT_ROOT / "Kjósendakönnun" / "ÍSKOS kjósenda 2024" / "ISKOS2024_med_aukabreytum_20260318.sav"
OUT_DIR = PROJECT_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Deterministic display width for the tables we print
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

print(f"Project root : {PROJECT_ROOT}")
print(f"Data file    : {DATA_PATH.name}")
print(f"Output dir   : {OUT_DIR}")

## 1 — Hlaða inn gögnum

Gögnin eru á SPSS-sniði (`.sav`). Við notum `pyreadstat` sem les bæði gögnin sjálf og lýsigögnin (spurningartexta og hvað gildin tákna). Við höldum tölulegum kóðum í gagnatöflunni (`apply_value_formats=False`) en nýtum lýsigögnin til að merkja myndir með íslenskum texta síðar.

In [ ]:
# Read the .sav; keep numeric codes for modeling (apply_value_formats=False).
# The metadata object `meta` holds variable labels (questions in Icelandic) and
# value labels (e.g., party code 1 -> "Samfylkinguna (S)") — we will reuse it
# for figure labels and to build target_labels.json below.
df_raw, meta = pyreadstat.read_sav(str(DATA_PATH), apply_value_formats=False)

print(f"Raw shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]:,} columns")
df_raw.head()

## 2 — Skilgreinum breytuhópa

Við veljum breytur í fjóra hópa. Fyrstu þrír hóparnir mynda **grunnlíkanið**, og *málefnaeign* bætist við til að mynda **útvíkkaða líkanið** síðar (annað skjalið keyrir bæði til samanburðar).

- **Félagsleg staða** (*social status*): lýðfræðilegar og efnahagslegar breytur sem eru stöðugar til meðallangs tíma.
- **Málefnaviðhorf** (*issue attitudes*): sammála/ósammála spurningar um stefnumál úr könnuninni.
- **Frammistaða stjórnvalda** (*government performance*): mat á síðustu ríkisstjórn og efnahag.
- **Málefnaeign** (*issue ownership*): hvaða flokki treystir svarandi best í tilteknu málefni. *Athugið:* þessar breytur eru mjög nálægt flokksvali, þær munu þar af leiðandi yfirgnæfa hinar í líkaninu. Við tökum þær út í aðalkeyrslunni og förum yfir hvað gerist þegar við bætum þeim inn.

Athugið að **flokkshollusta og afleiddar breytur um mat á flokkum og leiðtogum** (t.d. `prtfeel_*`, `ldrfeel_*`, `pid*`) eru vísvitandi **ekki** notaðar því þær eru jafn nálægt útkomunni og flokksvalið sjálft.

In [ ]:
# Feature groups (variable names in the .sav).
FEATURE_GROUPS = {
    "felagsleg_stada": [
        "gender", "age", "educationflo", "elecdist", "urban", "emplstatus",
        "incomecate", "rentown", "maritals", "hhmmb", "childs", "languagecat",
        "church", "rlgatnd",
    ],
    "malefnavidhorf": [
        "lrscale", "polintr", "pltrust",
        "envirimp", "gincdif", "taxreduc", "capicntry", "ceqvote", "privhealth",
        "imueclt", "essfull", "femfight", "banks", "immrght", "genconsind",
        "acrproimp", "drugpol", "gndequ", "EUmship",
    ],
    "frammistada_stjornvalda": ["govprfm", "ecobettwors_1"],
    "malefnaeign": [
        "bprteco", "bprtimm", "bprteurope", "bprtfamily", "bprtisencit",
        "bprthealth", "bprtenvir", "bprtregpol", "bprtedu", "bprtequal",
        "bprthouse",
    ],
}

# Sanity-check: every listed variable exists in the dataframe
missing = [v for group in FEATURE_GROUPS.values() for v in group if v not in df_raw.columns]
assert not missing, f"Expected columns not in data: {missing}"
print(f"Total features across all four groups: "
      f"{sum(len(v) for v in FEATURE_GROUPS.values())}")

# Which variables we still have to flag as categorical (nominal, no inherent order)
# These get one-hot encoded inside notebook 02; the rest are treated as numeric
# (ordinal Likert variables use numeric dtype so tree splits behave naturally).
NOMINAL_COLS = {
    "gender", "educationflo", "elecdist", "urban", "emplstatus",
    "rentown", "maritals", "languagecat", "church",
    # Issue-ownership variables are party-choice -> nominal
    "bprteco", "bprtimm", "bprteurope", "bprtfamily", "bprtisencit",
    "bprthealth", "bprtenvir", "bprtregpol", "bprtedu", "bprtequal", "bprthouse",
}

## 3 — Skilgreining útkomubreyta (7 flokkar)

Úrslit Alþingiskosninganna 2024 voru á þann veg að **sex flokkar** fengu fólk á þing: Samfylkingin (S), Framsóknarflokkurinn (B), Sjálfstæðisflokkurinn (D), Viðreisn (C), Miðflokkurinn (M), Flokkur fólksins (F). **Þrír flokkar til viðbótar** (Sósíalistaflokkur J, Píratar P, Vinstri græn V) fengu 2–4% atkvæða hver en engan þingmann.

Þar sem viðhorf bæði kjósenda og frambjóðenda þessara þriggja flokka eru um margt lík sameinum við svörin í einn flokk, `Vinstri_sameinad`. Þannig er útkomubreytan með **sjö mögulegir flokkar**.

Eingöngu þau sem **kusu og gáfu upp flokk** eru með — við sleppum:
- svarendum sem kusu ekki (`vote24 ≠ 1`)
- svörum *Man það ekki / Neitar að svara / Annað / Blanki / Lýðræðisflokk / Ábyrg framtíð* (ekki meðal okkar sjö flokka)

In [ ]:
# Party codes -> short key used internally
# (matches the prtvote24 value labels from the SPSS metadata)
PARTY_MAP = {
    1.0:  "S",   # Samfylkingin
    2.0:  "B",   # Framsókn
    3.0:  "D",   # Sjálfstæðis
    4.0:  "V",   # VG            -> merged
    6.0:  "P",   # Píratar       -> merged
    8.0:  "C",   # Viðreisn
    10.0: "M",   # Miðflokkur
    11.0: "F",   # Flokkur fólksins
    12.0: "J",   # Sósíalistar   -> merged
}
LEFT_MERGE = {"J", "P", "V"}  # these three get collapsed into one class
TARGET_KEEP_CODES = set(PARTY_MAP.keys())  # everything else is filtered out

# Icelandic display labels for the 7 target classes (used on figure axes later)
TARGET_LABELS_IS = {
    "S": "Samfylkingin",
    "B": "Framsóknarflokkur",
    "D": "Sjálfstæðisflokkur",
    "C": "Viðreisn",
    "M": "Miðflokkur",
    "F": "Flokkur fólksins",
    "Vinstri_sameinad": "Vinstri (sameinað: VG + Píratar + Sósíalistar)",
}

def build_target(series: pd.Series) -> pd.Series:
    """Map prtvote24 numeric codes to one of 7 short class keys, or NaN."""
    # Stage 1: numeric code -> short party key (or NaN for codes we don't keep)
    short = series.map(PARTY_MAP)
    # Stage 2: collapse J/P/V into a single merged class
    return short.where(~short.isin(LEFT_MERGE), other="Vinstri_sameinad")

# Apply filter: (a) respondent reported voting, (b) prtvote24 is in our kept set
voted = df_raw["vote24"] == 1.0
has_party = df_raw["prtvote24"].isin(TARGET_KEEP_CODES)
mask = voted & has_party

y = build_target(df_raw.loc[mask, "prtvote24"])
assert y.isna().sum() == 0, "target must be fully resolved after filtering"
assert y.nunique() == 7, f"expected 7 target classes, got {y.nunique()}"

print(f"Respondents kept: {mask.sum():,} of {len(df_raw):,}")
print("Class distribution:")
print(y.value_counts().rename("n").to_frame().assign(pct=lambda d: (d["n"] / d["n"].sum() * 100).round(1)))

## 4 — Undirbúum gögnin í töflu

Við búum til eina gagnatöflu `X` með **öllum** völdum forspárbreytum (bæði grunn- og útvíkkuðum). Annað skjalið setur saman grunn-fylkið (án málefnaeignar) og útvíkkaða fylkið (með málefnaeign) út frá `feature_groups.json`. Þessi fylki eru svo notuð til að mata líkönin með gögnum við þjálfun.

Við merkjum **nafnbreytur** (t.d. kjördæmi, kirkju, flokksheiti) sem `pandas.Categorical`. Slíkar breytur fá **one-hot kóðun** þegar líkönin eru þjálfuð, þ.e., við búum til dálk fyrir hvert og eitt mögulegt nafngildi og gildin í öllum dálkum eru núll nema í þeim dálki sem passar við gildi breytunnar. **Raðbreytur** (t.d. Likert-kvarðinn um sammála/sammála) halda sér sem tölulegar m.a. því ákvarðanatré geta þá klofið þær á náttúrulegum röðunargildum.

In [ ]:
# Collect all feature columns in a fixed order (base first, enriched appended)
base_cols = (
    FEATURE_GROUPS["felagsleg_stada"]
    + FEATURE_GROUPS["malefnavidhorf"]
    + FEATURE_GROUPS["frammistada_stjornvalda"]
)
enriched_cols = base_cols + FEATURE_GROUPS["malefnaeign"]

X = df_raw.loc[mask, enriched_cols].copy()

# Convert nominal columns to pandas Categorical. We use a "Missing" level
# instead of NaN so downstream OneHotEncoder doesn't need a separate imputation step.
for col in NOMINAL_COLS & set(X.columns):
    # Round numeric codes to int where possible (SPSS stores them as float);
    # keep them as strings so the one-hot encoder produces readable column names.
    col_as_str = X[col].apply(lambda v: f"{int(v)}" if pd.notna(v) else "Missing")
    X[col] = pd.Categorical(col_as_str)

# Numeric columns: keep as float64 but confirm dtype
numeric_cols = [c for c in X.columns if c not in NOMINAL_COLS]
X[numeric_cols] = X[numeric_cols].astype(float)

print(f"Feature matrix shape: {X.shape}")
print(f"  numeric cols : {len(numeric_cols)}")
print(f"  nominal cols : {len(NOMINAL_COLS & set(X.columns))}")
X.dtypes.head(20)

## 5 — Eyður í gögnum: hvernig við meðhöndlum þær og hvað mætti gera öðruvísi

ÍSKOS 2024 notar **einingauppbyggingu**: sérhver svarandi svarar kjarnaspurningalistanum og einni eða fleiri *einingum* en sleppir öðrum. Það þýðir að breytur utan kjarnans eru kerfisbundið ekki til staðar hjá hluta úrtaksins.

Við veljum einfalda, gagnsæja og vel skjalfesta meðhöndlun:

1. **Töluleg gildi** → miðgildi notað (*median imputation*) + aukabreyta `<var>_was_missing` fyrir hverja breytu þar sem yfir 5% gilda vantar.
2. **Nafnbreytur** → nýr flokkur `"Missing"` sem er notaður þegar breytuna vantar.

Við gerum engar beinar breytingar hér en gerum þær frekar í *forvinnsluhlutanum* (annað skjalið). Þannig er miðgildið **reiknað eingöngu á þjálfunarhluta** og beitt á prófunarhluta. Það kemur í veg fyrir *gagnaleka* (e. data leakage) því upplýsingar um prófunargögn mega ekki leka inn í þjálfunargögnin.

In [ ]:
# Report missingness for transparency — the imputation itself happens in notebook 02.
missing_rate = X.isna().mean().sort_values(ascending=False)
high_missing = missing_rate[missing_rate > 0.05]

print(f"Columns with >5% missingness ({len(high_missing)} of {len(missing_rate)}):")
print(high_missing.round(3).to_frame("missing_rate"))

## 6 — Skipting í þjálfunar- og prófunarhluta (80/20, lagskipt)

Við notum **lagskipta** (*stratified*) skiptingu vegna ójafnrar flokkadreifingar. Samfylkingin er með ~25% en smærri flokkar hafa <10%. Lagskipting tryggir að hlutföll haldist svipuð í báðum hlutum.

`random_state=42` tryggir að sama skipting verði til í hvert skipti sem þessi kóði er keyrður.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {len(X_train):,} rows")
print(f"Test : {len(X_test):,} rows")
print()
print("Class distribution (train vs test, %):")
train_pct = y_train.value_counts(normalize=True).mul(100).round(1)
test_pct = y_test.value_counts(normalize=True).mul(100).round(1)
pd.concat([train_pct.rename("train%"), test_pct.rename("test%")], axis=1)

## 7 — Prófanir á útkomunni

Við staðfestum þrjár forsendur áður en við vistum gögnin:

1. Útkomubreytan hefur nákvæmlega **7 flokka**.
2. Dálkar í `X_train` og `X_test` eru **eins** (í sömu röð).
3. **Enginn svarandi lendir í báðum hlutum** (engin samfella milli þjálfunar
   og prófunar).

In [ ]:
# (1) Target has exactly 7 classes
assert y_train.nunique() == 7 and y_test.nunique() == 7, (
    "target must have 7 classes in both splits"
)

# (2) Column lists match between train and test
assert list(X_train.columns) == list(X_test.columns), (
    "train/test feature columns must be identical and in the same order"
)

# (3) No index overlap (User_Id is not a column we kept, so we use the pandas index,
# which is inherited from the original dataframe row index).
assert X_train.index.intersection(X_test.index).empty, (
    "train and test must not share respondents"
)

print("All sanity checks passed.")

## 8 — Vista gögn og lýsigögn

Við vistum þjálfunar- og prófunargögnin á **parquet-sniði** (en ekki CSV), því parquet varðveitir pandas-gagnagerðir, þar á meðal `Categorical`, sem skiptir máli fyrir one-hot kóðunina í næsta skjali.

Við skrifum einnig tvær JSON-skrár:

- **`feature_groups.json`** — skrá yfir hvaða breytur tilheyra hvaða hópi og hverjar þeirra eru nafnbreytur (fyrir one-hot) eða tölulegar.
- **`target_labels.json`** — vörpun frá stuttum flokkslykli yfir í íslenskt flokksheiti, auk vörpunar frá breytukóða yfir í íslenskan spurningartexta úr SPSS-lýsigögnum (notað til að merkja myndir í seinni skjölum).

In [ ]:
# Training and test frames with target as a column named "y" — simpler downstream
train_df = X_train.copy()
train_df["y"] = y_train.values
test_df = X_test.copy()
test_df["y"] = y_test.values

train_path = OUT_DIR / "train.parquet"
test_path = OUT_DIR / "test.parquet"
train_df.to_parquet(train_path, index=True)
test_df.to_parquet(test_path, index=True)
print(f"Wrote {train_path} ({train_df.shape})")
print(f"Wrote {test_path} ({test_df.shape})")

In [ ]:
# feature_groups.json: which variables live in which group + nominal vs numeric
feature_groups_out = {
    "groups": FEATURE_GROUPS,
    "base_features": base_cols,
    "enriched_features": enriched_cols,
    "nominal_columns": sorted(NOMINAL_COLS & set(X.columns)),
    "numeric_columns": [c for c in X.columns if c not in NOMINAL_COLS],
}
groups_path = OUT_DIR / "feature_groups.json"
with groups_path.open("w", encoding="utf-8") as f:
    json.dump(feature_groups_out, f, ensure_ascii=False, indent=2)
print(f"Wrote {groups_path}")

In [ ]:
# target_labels.json: target codes + Icelandic variable descriptions
# meta.column_names_to_labels is a dict: variable-code -> question text (Icelandic).
variable_labels = {
    col: meta.column_names_to_labels.get(col, col)
    for col in enriched_cols
}

# Value labels for each nominal variable (SPSS-level value labels, e.g. party codes)
# We store these to annotate one-hot feature names as human-readable labels later.
value_labels = {}
for col in (NOMINAL_COLS & set(enriched_cols)):
    labs = meta.variable_value_labels.get(col, {})
    # Keys come in as floats -> stringify so JSON is well-formed
    value_labels[col] = {str(int(k)) if float(k).is_integer() else str(k): v for k, v in labs.items()}

target_labels_out = {
    "target_class_labels_is": TARGET_LABELS_IS,
    "variable_labels_is": variable_labels,
    "value_labels_is": value_labels,
}
target_path = OUT_DIR / "target_labels.json"
with target_path.open("w", encoding="utf-8") as f:
    json.dump(target_labels_out, f, ensure_ascii=False, indent=2)
print(f"Wrote {target_path}")

## Samantekt

Við höfum nú **hreinsað og skipt** gögnunum svo þau séu tilbúin fyrir líkanagerð.

| Skrá | Lýsing |
|---|---|
| `data/processed/train.parquet` | 80% af svarendum, með forspárbreytum og útkomunni `y` |
| `data/processed/test.parquet` | 20% af svarendum, sama snið |
| `data/processed/feature_groups.json` | Breytur flokkaðar í hópa; nafn- vs. tölulegt |
| `data/processed/target_labels.json` | Íslensk heiti á flokkum og breytum fyrir myndir |

**Næst:** `02_models.ipynb` hleður þessum skrám, þjálfar sex líkön (tvö viðmiðunarlíkön,
ákvarðanatökutré, random forest, gradient boosting og AutoGluon) og ber þau saman.